# ONNX Runtime GPU vs PyTorch: Performance & Memory Benchmark

This notebook compares the performance of running the DAS model using ONNX Runtime GPU versus converting it to PyTorch. We'll measure inference time for 8192 predictions and monitor VRAM usage.

## 1. Import Required Libraries

In [1]:
import os
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# ONNX Runtime
import onnxruntime as ort

# PyTorch and ONNX conversion
import torch
import onnx
from onnx2pytorch import ConvertModel

# Memory monitoring
import psutil
import GPUtil

# Visualization
import matplotlib.pyplot as plt
import pandas as pd

print("NumPy:", np.__version__)
print("ONNX Runtime:", ort.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA Device:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

ModuleNotFoundError: No module named 'torch'

## 2. Load ONNX Model with ONNX Runtime

In [ ]:
onnx_path = "das_model.onnx"

# Define execution providers with CUDA first, fallback to CPU
providers = [
    ("CUDAExecutionProvider", {
        "arena_extend_strategy": "kNextPowerOfTwo",
        "do_copy_in_default_stream": 1,
    }),
    "CPUExecutionProvider",
]

# Load ONNX model
ort_sess = ort.InferenceSession(onnx_path, providers=providers)

print("ONNX Runtime Session Created")
print("Execution Providers in use:", ort_sess.get_providers())

# Get model input/output info
ort_input = ort_sess.get_inputs()[0]
ort_output = ort_sess.get_outputs()[0]

print("\nModel Input:")
print(f"  Name: {ort_input.name}")
print(f"  Shape: {ort_input.shape}")
print(f"  Type: {ort_input.type}")

print("\nModel Output:")
print(f"  Name: {ort_output.name}")
print(f"  Shape: {ort_output.shape}")
print(f"  Type: {ort_output.type}")

# Determine input shape
B = 1
T = ort_input.shape[1]
C = ort_input.shape[2]

# Handle dynamic shapes
if isinstance(T, str) or T is None:
    T = 8192
if isinstance(C, str) or C is None:
    C = 1

print(f"\nUsing input shape: ({B}, {T}, {C})")

## 3. Convert ONNX Model to PyTorch

In [ ]:
# Load ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model loaded and checked")

# Convert to PyTorch
pytorch_model = ConvertModel(onnx_model, experimental=True)
print("ONNX model converted to PyTorch")

# Move to GPU
if torch.cuda.is_available():
    pytorch_model = pytorch_model.cuda()
    print("PyTorch model moved to GPU")
else:
    print("Warning: CUDA not available, using CPU")

# Set to eval mode
pytorch_model.eval()
print("PyTorch model set to eval mode")

## 4. Benchmark Inference Time: ONNX Runtime

In [ ]:
# Warmup
print("Warming up ONNX Runtime...")
for _ in range(3):
    x_dummy = np.random.randn(B, T, C).astype(np.float32)
    _ = ort_sess.run(None, {ort_input.name: x_dummy})

# Benchmark with 8192 predictions
num_runs = 10
print(f"\nBenchmarking ONNX Runtime with {num_runs} runs...")

ort_times = []
x_test = np.random.randn(B, T, C).astype(np.float32)

torch.cuda.synchronize() if torch.cuda.is_available() else None

for i in range(num_runs):
    start = time.perf_counter()
    output = ort_sess.run(None, {ort_input.name: x_test})
    end = time.perf_counter()
    ort_times.append(end - start)
    
    if (i + 1) % 5 == 0:
        print(f"  Run {i + 1}/{num_runs} - Time: {ort_times[-1]*1000:.2f}ms")

torch.cuda.synchronize() if torch.cuda.is_available() else None

ort_avg_time = np.mean(ort_times)
ort_min_time = np.min(ort_times)
ort_max_time = np.max(ort_times)
ort_std_time = np.std(ort_times)

print(f"\nONNX Runtime Results:")
print(f"  Average Time: {ort_avg_time*1000:.4f}ms")
print(f"  Min Time:     {ort_min_time*1000:.4f}ms")
print(f"  Max Time:     {ort_max_time*1000:.4f}ms")
print(f"  Std Dev:      {ort_std_time*1000:.4f}ms")
print(f"  Output shape: {output[0].shape}")

## 5. Benchmark Inference Time: PyTorch

In [ ]:
# Warmup
print("Warming up PyTorch...")
with torch.no_grad():
    for _ in range(3):
        x_dummy = torch.randn(B, T, C, dtype=torch.float32)
        if torch.cuda.is_available():
            x_dummy = x_dummy.cuda()
        _ = pytorch_model(x_dummy)

# Benchmark with 8192 predictions
num_runs = 10
print(f"\nBenchmarking PyTorch with {num_runs} runs...")

pytorch_times = []
x_test_torch = torch.randn(B, T, C, dtype=torch.float32)
if torch.cuda.is_available():
    x_test_torch = x_test_torch.cuda()

torch.cuda.synchronize() if torch.cuda.is_available() else None

with torch.no_grad():
    for i in range(num_runs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        
        start = time.perf_counter()
        output_pytorch = pytorch_model(x_test_torch)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        pytorch_times.append(end - start)
        
        if (i + 1) % 5 == 0:
            print(f"  Run {i + 1}/{num_runs} - Time: {pytorch_times[-1]*1000:.2f}ms")

torch.cuda.synchronize() if torch.cuda.is_available() else None

pytorch_avg_time = np.mean(pytorch_times)
pytorch_min_time = np.min(pytorch_times)
pytorch_max_time = np.max(pytorch_times)
pytorch_std_time = np.std(pytorch_times)

print(f"\nPyTorch Results:")
print(f"  Average Time: {pytorch_avg_time*1000:.4f}ms")
print(f"  Min Time:     {pytorch_min_time*1000:.4f}ms")
print(f"  Max Time:     {pytorch_max_time*1000:.4f}ms")
print(f"  Std Dev:      {pytorch_std_time*1000:.4f}ms")
print(f"  Output shape: {output_pytorch.shape}")

## 6. Measure VRAM Usage: ONNX Runtime

In [ ]:
print("Measuring VRAM usage for ONNX Runtime...")

if torch.cuda.is_available():
    # Get initial GPU memory
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    initial_memory = torch.cuda.memory_allocated() / (1024**3)  # GB
    print(f"Initial VRAM: {initial_memory:.4f} GB")
    
    # Run inference and measure peak memory
    x_test = np.random.randn(B, T, C).astype(np.float32)
    
    output = ort_sess.run(None, {ort_input.name: x_test})
    
    peak_memory = torch.cuda.max_memory_allocated() / (1024**3)  # GB
    current_memory = torch.cuda.memory_allocated() / (1024**3)  # GB
    
    ort_vram_used = peak_memory - initial_memory
    
    print(f"Peak VRAM during inference: {peak_memory:.4f} GB")
    print(f"Current VRAM after inference: {current_memory:.4f} GB")
    print(f"VRAM used by ONNX Runtime: {ort_vram_used:.4f} GB")
    
    # Clean up
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
else:
    print("CUDA not available, cannot measure GPU VRAM")
    ort_vram_used = None

## 7. Measure VRAM Usage: PyTorch

In [ ]:
print("Measuring VRAM usage for PyTorch...")

if torch.cuda.is_available():
    # Get initial GPU memory
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    initial_memory = torch.cuda.memory_allocated() / (1024**3)  # GB
    print(f"Initial VRAM: {initial_memory:.4f} GB")
    
    # Run inference and measure peak memory
    x_test_torch = torch.randn(B, T, C, dtype=torch.float32).cuda()
    
    with torch.no_grad():
        output = pytorch_model(x_test_torch)
        torch.cuda.synchronize()
    
    peak_memory = torch.cuda.max_memory_allocated() / (1024**3)  # GB
    current_memory = torch.cuda.memory_allocated() / (1024**3)  # GB
    
    pytorch_vram_used = peak_memory - initial_memory
    
    print(f"Peak VRAM during inference: {peak_memory:.4f} GB")
    print(f"Current VRAM after inference: {current_memory:.4f} GB")
    print(f"VRAM used by PyTorch: {pytorch_vram_used:.4f} GB")
    
    # Clean up
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
else:
    print("CUDA not available, cannot measure GPU VRAM")
    pytorch_vram_used = None

## 8. Compare Results

In [ ]:
# Create comparison summary
print("\n" + "="*70)
print("PERFORMANCE COMPARISON SUMMARY")
print("="*70)

# Create DataFrame for easy comparison
comparison_data = {
    'Metric': [
        'Average Inference Time (ms)',
        'Min Inference Time (ms)',
        'Max Inference Time (ms)',
        'Std Dev (ms)',
        'VRAM Usage (GB)'
    ],
    'ONNX Runtime': [
        f"{ort_avg_time*1000:.4f}",
        f"{ort_min_time*1000:.4f}",
        f"{ort_max_time*1000:.4f}",
        f"{ort_std_time*1000:.4f}",
        f"{ort_vram_used:.4f}" if ort_vram_used else "N/A"
    ],
    'PyTorch': [
        f"{pytorch_avg_time*1000:.4f}",
        f"{pytorch_min_time*1000:.4f}",
        f"{pytorch_max_time*1000:.4f}",
        f"{pytorch_std_time*1000:.4f}",
        f"{pytorch_vram_used:.4f}" if pytorch_vram_used else "N/A"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + df_comparison.to_string(index=False))

# Calculate speedup
speedup = pytorch_avg_time / ort_avg_time if ort_avg_time > 0 else 0
print(f"\nSpeedup (PyTorch / ONNX Runtime): {speedup:.2f}x")
if speedup > 1:
    print(f"  → ONNX Runtime is {speedup:.2f}x FASTER")
else:
    print(f"  → PyTorch is {1/speedup:.2f}x FASTER")

# Calculate VRAM difference
if ort_vram_used and pytorch_vram_used:
    vram_diff = pytorch_vram_used - ort_vram_used
    vram_ratio = pytorch_vram_used / ort_vram_used if ort_vram_used > 0 else 0
    print(f"\nVRAM Difference: {vram_diff:.4f} GB ({vram_ratio:.2f}x)")
    if vram_diff > 0:
        print(f"  → ONNX Runtime uses {vram_diff:.4f} GB LESS VRAM")
    else:
        print(f"  → PyTorch uses {-vram_diff:.4f} GB LESS VRAM")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ONNX Runtime vs PyTorch: Performance Comparison', fontsize=16, fontweight='bold')

# 1. Average Inference Time
ax1 = axes[0, 0]
models = ['ONNX Runtime', 'PyTorch']
avg_times = [ort_avg_time*1000, pytorch_avg_time*1000]
colors = ['#3498db', '#e74c3c']
bars1 = ax1.bar(models, avg_times, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Time (ms)', fontsize=11, fontweight='bold')
ax1.set_title('Average Inference Time', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for bar, time in zip(bars1, avg_times):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{time:.4f}ms', ha='center', va='bottom', fontweight='bold')

# 2. Min vs Max Time
ax2 = axes[0, 1]
x = np.arange(len(models))
width = 0.35
min_times = [ort_min_time*1000, pytorch_min_time*1000]
max_times = [ort_max_time*1000, pytorch_max_time*1000]
bars2_1 = ax2.bar(x - width/2, min_times, width, label='Min Time', color='#2ecc71', alpha=0.7, edgecolor='black')
bars2_2 = ax2.bar(x + width/2, max_times, width, label='Max Time', color='#e67e22', alpha=0.7, edgecolor='black')
ax2.set_ylabel('Time (ms)', fontsize=11, fontweight='bold')
ax2.set_title('Min vs Max Inference Time', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(models)
ax2.legend(loc='upper left')
ax2.grid(axis='y', alpha=0.3)

# 3. VRAM Usage
if ort_vram_used and pytorch_vram_used:
    ax3 = axes[1, 0]
    vram_usage = [ort_vram_used, pytorch_vram_used]
    bars3 = ax3.bar(models, vram_usage, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax3.set_ylabel('VRAM (GB)', fontsize=11, fontweight='bold')
    ax3.set_title('GPU VRAM Usage', fontsize=12, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    for bar, vram in zip(bars3, vram_usage):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                 f'{vram:.4f}GB', ha='center', va='bottom', fontweight='bold')

# 4. Inference Time Distribution
ax4 = axes[1, 1]
ax4.hist(ort_times, bins=10, alpha=0.6, label='ONNX Runtime', color='#3498db', edgecolor='black')
ax4.hist(pytorch_times, bins=10, alpha=0.6, label='PyTorch', color='#e74c3c', edgecolor='black')
ax4.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax4.set_title('Inference Time Distribution', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization complete!")

## Summary & Recommendations

### Key Findings:
- **Inference Speed**: Compare the average inference times to determine which framework is faster
- **VRAM Usage**: Analyze which framework uses less GPU memory
- **Consistency**: Check the standard deviation to see which framework has more stable performance

### Considerations:
1. **ONNX Runtime**: Generally more optimized for inference, faster and uses less memory
2. **PyTorch**: More flexible for research and development, may be slower but easier to modify
3. **Input Size**: For 8192 samples (as tested), performance characteristics may vary with different input sizes

### Next Steps:
- Test with different batch sizes to see how each framework scales
- Profile memory usage in more detail if needed
- Consider deployment size and dependencies when choosing framework